# Model Trainging – Heart Disease Dataset
*Training and evaluation of models for Cardio - Risk Prediction project*  

---

## Table of Contents


<a id='imports'></a>
## Reproducibility & Imports  
---

In [1]:
# Reproducibility
import os, sys, itertools, random, numpy as np
import joblib, pandas as pd

# Imports
sys.path.append(os.path.abspath(os.path.join('..')))
sys.path.append(os.path.abspath(os.path.join('..', 'scripts')))

# Models & tools
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, StackingClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from scripts.model_training.utils import evaluate_classification, _score_metric, _oof_scores
from sklearn.base import clone
from sklearn.metrics import roc_curve, precision_recall_curve, auc
import matplotlib.pyplot as plt

SEED = 42
OUTPUT_DIR = "../outputs/models"

In [2]:
df = joblib.load("../outputs/eda/df_preprocessed_balanced.pkl")

In [3]:
df_copy = df.copy()
TARGET_COL = 'DEATH_EVENT'

X = df_copy.iloc[:, :-1]
y = df_copy.iloc[:, -1]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

Data split

In [4]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Split shapes:")
print("  X_tr:", X_tr.shape, "| X_te:", X_te.shape)

Split shapes:
  X_tr: (246, 12) | X_te: (62, 12)


<a id='random-forest'></a>
---
## Random Forest

In [5]:
rf_params_grid = {
    'n_estimators': [600, 700, 800, 900, 1000],
    'max_depth': [5, 6, 7],
    'min_samples_split': [2, 3, 4, 6, 8],
    'min_samples_leaf': [1, 2, 3],
    'max_features': ['sqrt', 0.4, 0.6],
    'bootstrap': [True, False],
    'class_weight': [None, 'balanced'],
}

rf_model = RandomForestClassifier(
    n_jobs=-1, 
    random_state=SEED
)

rf_grid = GridSearchCV(
    estimator = rf_model,
    param_grid = rf_params_grid,
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring = 'accuracy',
    n_jobs = -1,
    verbose = 10,
    refit=True,
)

In [ ]:
rf_grid.fit(X_tr, y_tr)
rf_model = rf_grid.best_estimator_

print('Best parameters:', rf_grid.best_params_)
df_metrics, details = evaluate_classification(rf_model, X_te, y_te)

# save the best model
os.makedirs(OUTPUT_DIR, exist_ok=True)
joblib.dump(rf_model, os.path.join(OUTPUT_DIR, 'rf.pkl'))

Fitting 5 folds for each of 2700 candidates, totalling 13500 fits


<a id='adaboost'></a>
## AdaBoost

In [ ]:
ada_params_grid = {
    'n_estimators': [50, 100, 200, 300, 600, 700, 800, 900, 1000],
    'learning_rate': [0.01, 0.05, 0.1],
    'estimator__max_depth': [1, 2, 3, 4, 5, 6, 7],
    'estimator__min_samples_leaf': [1, 2, 3],
    'estimator__class_weight': [None, 'balanced'],
}

ada_estimator = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(random_state=SEED), 
    random_state=SEED
)

ada_grid = GridSearchCV(
    estimator = ada_estimator,
    param_grid = ada_params_grid,
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring = 'accuracy',
    n_jobs = -1,
    verbose = 10,
    refit=True,
)

In [ ]:
ada_grid.fit(X_tr, y_tr)
ada_model = ada_grid.best_estimator_

print('Best params:', ada_grid.best_params_)
df_metrics, details = evaluate_classification(ada_model, X_te, y_te)

os.makedirs(OUTPUT_DIR, exist_ok=True)
joblib.dump(ada_model, os.path.join(OUTPUT_DIR, 'ada.pkl'))

Fitting 5 folds for each of 1134 candidates, totalling 5670 fits
Best params: {'estimator__class_weight': None, 'estimator__max_depth': 3, 'estimator__min_samples_leaf': 2, 'learning_rate': 0.1, 'n_estimators': 900}


,Metric,Value
0,Accuracy,0.9032
1,Balanced accuracy,0.8804
2,Precision (pos=1),0.8947
3,Recall / Sensitivity (TPR),0.8095
4,Specificity (TNR),0.9512
5,F1,0.8500
6,F0.5,0.8763
7,F2,0.8252
8,ROC-AUC,0.9686
9,PR-AUC (Average Precision),0.9414



Confusion matrix:


,Pred 0,Pred 1
Actual 0,39,2
Actual 1,4,17


['../outputs/models\\ada.pkl']

<a id='xgboost'></a>
## XGBoost


In [ ]:
xgb_params_grid = {
    'n_estimators': [100, 200, 300, 400, 500],   
    'learning_rate': [0.001, 0.01, 0.05, 0.1],
    'max_depth': [2, 3, 4],
    'min_child_weight': [1, 2, 4, 6, 7],
    'subsample': [0.5, 1.0],
    'reg_lambda': [0.01, 0.05, 0.5],
}

xgb_estimator = XGBClassifier(
    tree_method = 'hist',
    objective = 'binary:logistic',
    eval_metric = 'logloss',
    n_jobs = -1,
    random_state = SEED,
)

xgb_grid = GridSearchCV(
    estimator = xgb_estimator,
    param_grid = xgb_params_grid,
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring = 'accuracy',
    n_jobs = -1,
    
    verbose = 10,
    refit=True,
)

In [ ]:
xgb_grid.fit(X_tr, y_tr)
xgb_model = xgb_grid.best_estimator_

print('Best params:', xgb_grid.best_params_)
df_metrics, details = evaluate_classification(xgb_model, X_te, y_te)

os.makedirs(OUTPUT_DIR, exist_ok=True)
joblib.dump(xgb_model, os.path.join(OUTPUT_DIR, 'xgb.pkl'))

Fitting 5 folds for each of 1800 candidates, totalling 9000 fits
Best params: {'learning_rate': 0.05, 'max_depth': 2, 'min_child_weight': 1, 'n_estimators': 400, 'reg_lambda': 0.05, 'subsample': 1.0}


,Metric,Value
0,Accuracy,0.9194
1,Balanced accuracy,0.9042
2,Precision (pos=1),0.9000
3,Recall / Sensitivity (TPR),0.8571
4,Specificity (TNR),0.9512
5,F1,0.8780
6,F0.5,0.8911
7,F2,0.8654
8,ROC-AUC,0.9652
9,PR-AUC (Average Precision),0.9133



Confusion matrix:


,Pred 0,Pred 1
Actual 0,39,2
Actual 1,3,18


['../outputs/models\\xgb.pkl']

## Naive Bayes

In [ ]:
nb_params_grid = {
    "var_smoothing": np.logspace(-12, -6, 13),
}

nb_estimator = GaussianNB()

nb_grid = GridSearchCV(
    estimator=nb_estimator,
    param_grid=nb_params_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring="accuracy",
    n_jobs=-1,
    verbose=10,
)

In [ ]:
nb_grid.fit(X_tr, y_tr)
nb_model = nb_grid.best_estimator_

print('Best params:', nb_grid.best_params_)
df_metrics, details = evaluate_classification(nb_model, X_te, y_te)

os.makedirs(OUTPUT_DIR, exist_ok=True)
joblib.dump(nb_model, os.path.join(OUTPUT_DIR, 'naive_bayes.pkl'))

Fitting 5 folds for each of 13 candidates, totalling 65 fits
Best params: {'var_smoothing': np.float64(1e-12)}


,Metric,Value
0,Accuracy,0.8871
1,Balanced accuracy,0.8682
2,Precision (pos=1),0.8500
3,Recall / Sensitivity (TPR),0.8095
4,Specificity (TNR),0.9268
5,F1,0.8293
6,F0.5,0.8416
7,F2,0.8173
8,ROC-AUC,0.9303
9,PR-AUC (Average Precision),0.8441



Confusion matrix:


,Pred 0,Pred 1
Actual 0,38,3
Actual 1,4,17


['../outputs/models\\naive_bayes.pkl']

## SVM

In [ ]:
svm_params_grid = {
    "kernel": ["rbf", "linear", "poly"],
    "C": [0.1, 1, 5, 7, 10],
    "gamma": ["scale", "auto"],
    "class_weight": [None, "balanced"],
    "degree": [2, 3, 4],
}

svm_estimator = SVC(
    probability=True, 
    random_state=SEED
)

svm_grid = GridSearchCV(
    estimator = svm_estimator,
    param_grid = svm_params_grid,
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring = "accuracy",
    n_jobs = -1,
    verbose = 10,
    refit=True,
)

In [ ]:
svm_grid.fit(X_tr, y_tr)
svm_model = svm_grid.best_estimator_

print('Best params:', svm_grid.best_params_)
df_metrics, details = evaluate_classification(svm_model, X_te, y_te)

os.makedirs(OUTPUT_DIR, exist_ok=True)
joblib.dump(svm_model, os.path.join(OUTPUT_DIR, 'svm.pkl'))

Fitting 5 folds for each of 180 candidates, totalling 900 fits
Best params: {'C': 1, 'class_weight': None, 'degree': 3, 'gamma': 'auto', 'kernel': 'poly'}


,Metric,Value
0,Accuracy,0.9032
1,Balanced accuracy,0.8920
2,Precision (pos=1),0.8571
3,Recall / Sensitivity (TPR),0.8571
4,Specificity (TNR),0.9268
5,F1,0.8571
6,F0.5,0.8571
7,F2,0.8571
8,ROC-AUC,0.9489
9,PR-AUC (Average Precision),0.9201



Confusion matrix:


,Pred 0,Pred 1
Actual 0,38,3
Actual 1,3,18


['../outputs/models\\svm.pkl']

## Logistic Classifier

In [ ]:
log_params_grid = {
    "penalty": [None, "l2"],
    "C": np.logspace(-3, 3, 7),
    "class_weight": [None, "balanced"],
}

log_estimator = LogisticRegression(
    solver="lbfgs", 
    max_iter=5000, 
    random_state=SEED
)

log_grid = GridSearchCV(
    estimator = log_estimator,
    param_grid = log_params_grid,
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring = "accuracy",
    n_jobs = -1,
    verbose = 10,
)

In [ ]:
log_grid.fit(X_tr, y_tr)
log_model = log_grid.best_estimator_

print("Best params:", log_grid.best_params_)
df_metrics, details = evaluate_classification(log_model, X_te, y_te)

os.makedirs(OUTPUT_DIR, exist_ok=True)
joblib.dump(log_model, os.path.join(OUTPUT_DIR, 'logistic.pkl'))

Fitting 5 folds for each of 28 candidates, totalling 140 fits
Best params: {'C': np.float64(10.0), 'class_weight': None, 'penalty': 'l2'}


,Metric,Value
0,Accuracy,0.9194
1,Balanced accuracy,0.9042
2,Precision (pos=1),0.9000
3,Recall / Sensitivity (TPR),0.8571
4,Specificity (TNR),0.9512
5,F1,0.8780
6,F0.5,0.8911
7,F2,0.8654
8,ROC-AUC,0.9652
9,PR-AUC (Average Precision),0.9281



Confusion matrix:


,Pred 0,Pred 1
Actual 0,39,2
Actual 1,3,18


['../outputs/models\\logistic.joblib']

## Stacking

In [ ]:
rf_clone  = clone(rf_model)
ada_clone = clone(ada_model)
xgb_clone = clone(xgb_model)
svm_clone = clone(svm_model)
nb_best  = clone(nb_model)
log_clone = clone(log_model)

stacking_estimators = [
    ('rf', rf_clone),
    ('ada', ada_clone),
    ('xgb', xgb_clone),
    ('svm', svm_model),
    ("nb",  nb_best),
]

stacking_model = StackingClassifier(
    estimators=stacking_estimators,
    final_estimator=log_estimator,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    stack_method='predict_proba',
    n_jobs=-1
)

NameError: name 'ada_model' is not defined

In [ ]:
stacking_model.fit(X_tr, y_tr)
stack_df_metrics, stack_details = evaluate_classification(stacking_model, X_te, y_te)

os.makedirs(OUTPUT_DIR, exist_ok=True)
joblib.dump(stacking_model, os.path.join(OUTPUT_DIR, "stacking.pkl"))

,Metric,Value
0,Accuracy,0.9032
1,Balanced accuracy,0.8804
2,Precision (pos=1),0.8947
3,Recall / Sensitivity (TPR),0.8095
4,Specificity (TNR),0.9512
5,F1,0.8500
6,F0.5,0.8763
7,F2,0.8252
8,ROC-AUC,0.9768
9,PR-AUC (Average Precision),0.9518



Confusion matrix:


,Pred 0,Pred 1
Actual 0,39,2
Actual 1,4,17


['../outputs/models\\stacking.joblib']

## Voting

In [ ]:
voting_estimators = [
    ('rf', rf_clone),
    ('ada', ada_clone),
    ('xgb', xgb_clone),
    ('svm', svm_model),
    ("nb",  nb_best),
]

voting_model = VotingClassifier(
    estimators=voting_estimators,
    voting='soft',  # use probabilities
    weights=[2, 1, 2, 1, 1],
    n_jobs=-1,
)

In [ ]:
voting_model.fit(X_tr, y_tr)
df_metrics, details = evaluate_classification(voting_model, X_te, y_te)

os.makedirs(OUTPUT_DIR, exist_ok=True)
joblib.dump(voting_model, os.path.join(OUTPUT_DIR, 'voting.pkl'))

,Metric,Value
0,Accuracy,0.8871
1,Balanced accuracy,0.8682
2,Precision (pos=1),0.8500
3,Recall / Sensitivity (TPR),0.8095
4,Specificity (TNR),0.9268
5,F1,0.8293
6,F0.5,0.8416
7,F2,0.8173
8,ROC-AUC,0.9779
9,PR-AUC (Average Precision),0.9535



Confusion matrix:


,Pred 0,Pred 1
Actual 0,38,3
Actual 1,4,17


['../outputs/models\\voting.joblib']

### Weight search via CV ROC-AUC

In [ ]:
# --- 1) OOF probs once (no refits later) ---
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score
import numpy as np, itertools, pandas as pd

base_models = {
    "rf": rf_model,
    "ada": ada_model,
    "xgb": xgb_model,
    "svm": svm_model,
    "nb":  nb_model,
}
order = [k for k in ["rf","ada","xgb","svm","nb"] if k in base_models]

oof = {}
for k in order:
    m = base_models[k]
    if hasattr(m, "predict_proba"):
        oof[k] = cross_val_predict(m, X_tr, y_tr, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
    elif hasattr(m, "decision_function"):
        raw = cross_val_predict(m, X_tr, y_tr, cv=cv, method="decision_function", n_jobs=-1)
        oof[k] = (raw - raw.min()) / (raw.ptp() + 1e-12)
    else:
        oof[k] = cross_val_predict(m, X_tr, y_tr, cv=cv, method="predict", n_jobs=-1).astype(float)

# --- 2) Deterministic candidate set (no randomness) ---
candidates = [
    (rf, ada, xgb, svm, nb)
    for rf in range(1, 7)
    for ada in range(1, 7)
    for xgb in range(1, 7)
    for svm in range(0, 4)
    for nb  in range(0, 4)
    if rf + ada + xgb + svm + nb <= 16
]

# keep only the weights for the models we actually have (subset of order)
idx = {"rf":0,"ada":1,"xgb":2,"svm":3,"nb":4}
def project(w):
    return np.array([w[idx[k]] for k in order], dtype=float)

# --- 3) Score all candidates offline (ROC-AUC on OOF) ---
y_oof = y_tr.values if hasattr(y_tr, "values") else np.asarray(y_tr)
P = np.column_stack([oof[k] for k in order])  # shape (n_samples, n_models)

best_w, best_auc = None, -np.inf
rows = []
for w in candidates:
    wv = project(w)
    if wv.sum() == 0:  # should not happen with ranges above, but guard anyway
        continue
    p = (P @ wv) / wv.sum()  # weighted average of probs
    auc = roc_auc_score(y_oof, p)
    rows.append((w, auc))
    if auc > best_auc:
        best_auc, best_w = auc, w

rows.sort(key=lambda x: x[1], reverse=True)
print("Top candidates (by OOF ROC-AUC):")
for w, s in rows[:15]:
    print(f"  weights={w} -> OOF ROC-AUC={s:.4f}")
print(f"\n[SELECTED] weights={best_w} with OOF ROC-AUC={best_auc:.4f}")

# --- 4) Evaluate on test (single fit per model) ---
# get test probs once
test_probs = {}
for k, m in base_models.items():
    if hasattr(m, "predict_proba"):
        test_probs[k] = m.predict_proba(X_te)[:, 1]
    elif hasattr(m, "decision_function"):
        raw = m.decision_function(X_te)
        test_probs[k] = (raw - raw.min()) / (raw.ptp() + 1e-12)
    else:
        test_probs[k] = m.predict(X_te).astype(float)

wv = project(best_w)
p_test = (np.column_stack([test_probs[k] for k in order]) @ wv) / wv.sum()

# optional: tune threshold for the selected weights on OOF and then evaluate
from sklearn.metrics import precision_recall_curve
prec, rec, thr = precision_recall_curve(y_oof, (P @ wv) / wv.sum())
f1 = 2*prec*rec/(prec+rec+1e-12)
t_star = float(thr[np.nanargmax(f1[:-1])])  # best F1 threshold on OOF
print(f"Chosen threshold for ensemble (max OOF F1): {t_star:.3f}")

# Make a tiny wrapper to use evaluate_classification
class _FixedProba:
    def __init__(self, p): self.p = np.asarray(p)
    def predict_proba(self, X):  # X ignored; we pass stored probs
        return np.c_[1-self.p, self.p]

df_m, _ = evaluate_classification(_FixedProba(p_test), X_te, y_te, threshold=t_star, show=True)


In [ ]:
# random.seed(SEED)
# candidates = set()

# for rf_w, ada_w, xgb_w in itertools.product(range(1, 7), repeat=3):  # core models
#     for svm_w, nb_w in itertools.product(range(0, 4), repeat=2):     # aux models
#         if rf_w + ada_w + xgb_w + svm_w + nb_w <= 16:
#             candidates.add((rf_w, ada_w, xgb_w, svm_w, nb_w))

# emph = [
#     (k, 1, 1, 1, 1) for k in (3, 4, 5, 6)
# ] + [
#     (1, k, 1, 1, 1) for k in (3, 4, 5, 6)
# ] + [
#     (1, 1, k, 1, 1) for k in (3, 4, 5, 6)
# ] + [
#     (k, k-1, 1, 1, 0) for k in (3, 4, 5, 6)
# ] + [
#     (k, 1, k-1, 1, 0) for k in (3, 4, 5, 6)
# ] + [
#     (1, k, k-1, 1, 0) for k in (3, 4, 5, 6)
# ]
# for w in emph:
#     if all(x >= 0 for x in w) and sum(w) <= 16:
#         candidates.add(w)

# base = [
#     (1,1,1,0,0), (2,2,1,0,0), (3,2,2,0,0),
#     (4,2,2,0,0), (4,3,2,0,0), (5,3,2,0,0),
#     (2,1,2,1,0), (3,1,2,1,0), (3,2,1,1,0),
#     (2,2,2,1,1), (3,2,2,1,1), (4,2,2,1,1), (5,2,2,1,1),
# ]
# for w in base:
#     if sum(w) <= 16:
#         candidates.add(w)

# # randomized combos for diversity
# for _ in range(200):
#     rf = random.randint(1, 6)
#     ada = random.randint(1, 6)
#     xgb = random.randint(1, 6)
#     svm = random.randint(0, 3)
#     nb  = random.randint(0, 3)
#     w = (rf, ada, xgb, svm, nb)
#     if sum(w) <= 16:
#         candidates.add(w)

# candidates = sorted(candidates, key=lambda x: (sum(x), x))
# print(f"Total candidates generated: {len(candidates)}")

# best_w, best_auc = None, -np.inf
# scores = []

# for w in candidates:
#     vc = VotingClassifier(estimators=voting_estimators, voting="soft", weights=w, n_jobs=-1)
#     auc_cv = cross_val_score(vc, X_tr, y_tr, cv=cv, scoring="roc_auc", n_jobs=-1).mean()
#     scores.append((w, auc_cv))
#     if auc_cv > best_auc:
#         best_auc, best_w = auc_cv, w

# scores_sorted = sorted(scores, key=lambda x: x[1], reverse=True)
# print("\nTop candidates (by CV ROC-AUC):")
# for w, s in scores_sorted[:15]:
#     print(f"  weights={w} -> CV ROC-AUC={s:.4f}")

# print(f"\n[SELECTED] weights={best_w} with CV ROC-AUC={best_auc:.4f}")

In [ ]:
# voting_best = VotingClassifier(estimators=voting_estimators, voting="soft", weights=best_w, n_jobs=-1)
# print("\n[FIT] VotingClassifier (best weights)")
# voting_best.fit(X_tr, y_tr)

# print("\n[EVAL] VotingClassifier (best weights) on test")
# v_best_df_metrics, v_best_details = evaluate_classification(voting_best, X_te, y_te)
# joblib.dump(voting_best, os.path.join(OUTPUT_DIR, "voting_best.joblib"))


In [ ]:
# ('rf', 'ada', 'xgb', 'svm', 'nb')
candidates = [
    (1, 1, 1, 1, 1),   # equal
    (2, 1, 1, 1, 1),   # RF stronger
    (1, 2, 1, 1, 1),   # Ada stronger
    (1, 1, 2, 1, 1),   # XGB stronger
    (3, 2, 2, 1, 1),   # strong tree block (best found earlier)
    (4, 2, 2, 1, 1),
    (5, 2, 2, 1, 1),
    (3, 3, 2, 1, 1),
    (3, 2, 3, 1, 1),
    (4, 3, 2, 1, 1),
    (2, 6, 1, 1, 2),
    (2, 2, 2, 0, 0),   # no aux models
    (3, 3, 2, 0, 0),
    (4, 3, 2, 0, 0),
    (5, 3, 2, 0, 0)
]

best_w, best_auc = None, -np.inf
scores = []

for w in candidates:
    vc = VotingClassifier(estimators=voting_estimators, voting="soft", weights=w, n_jobs=-1)
    auc_cv = cross_val_score(vc, X_tr, y_tr, cv=cv, scoring="roc_auc", n_jobs=-1).mean()
    scores.append((w, auc_cv))
    if auc_cv > best_auc:
        best_auc, best_w = auc_cv, w

scores_sorted = sorted(scores, key=lambda x: x[1], reverse=True)
print("\nTop candidates (by CV ROC-AUC):")
for w, s in scores_sorted:
    print(f"  weights={w} -> CV ROC-AUC={s:.4f}")

print(f"\n[SELECTED] weights={best_w} with CV ROC-AUC={best_auc:.4f}")


Top candidates (by CV ROC-AUC):
  weights=(2, 6, 1, 1, 2) -> CV ROC-AUC=0.9735
  weights=(3, 3, 2, 0, 0) -> CV ROC-AUC=0.9720
  weights=(5, 3, 2, 0, 0) -> CV ROC-AUC=0.9720
  weights=(4, 3, 2, 0, 0) -> CV ROC-AUC=0.9717
  weights=(2, 2, 2, 0, 0) -> CV ROC-AUC=0.9706
  weights=(4, 2, 2, 1, 1) -> CV ROC-AUC=0.9698
  weights=(3, 2, 2, 1, 1) -> CV ROC-AUC=0.9698
  weights=(3, 3, 2, 1, 1) -> CV ROC-AUC=0.9694
  weights=(4, 3, 2, 1, 1) -> CV ROC-AUC=0.9691
  weights=(5, 2, 2, 1, 1) -> CV ROC-AUC=0.9691
  weights=(1, 2, 1, 1, 1) -> CV ROC-AUC=0.9690
  weights=(3, 2, 3, 1, 1) -> CV ROC-AUC=0.9687
  weights=(2, 1, 1, 1, 1) -> CV ROC-AUC=0.9683
  weights=(1, 1, 2, 1, 1) -> CV ROC-AUC=0.9675
  weights=(1, 1, 1, 1, 1) -> CV ROC-AUC=0.9672

[SELECTED] weights=(2, 6, 1, 1, 2) with CV ROC-AUC=0.9735


In [ ]:
voting_best = VotingClassifier(estimators=voting_estimators, voting="soft", weights=best_w, n_jobs=-1)
voting_best.fit(X_tr, y_tr)

df_metrics, details = evaluate_classification(voting_best, X_te, y_te)
joblib.dump(voting_best, os.path.join(OUTPUT_DIR, "voting_best.joblib"))

,Metric,Value
0,Accuracy,0.8871
1,Balanced accuracy,0.8566
2,Precision (pos=1),0.8889
3,Recall / Sensitivity (TPR),0.7619
4,Specificity (TNR),0.9512
5,F1,0.8205
6,F0.5,0.8602
7,F2,0.7843
8,ROC-AUC,0.9791
9,PR-AUC (Average Precision),0.9581



Confusion matrix:


,Pred 0,Pred 1
Actual 0,39,2
Actual 1,5,16


['../outputs/models\\voting_best.joblib']

## Threshold tuning

In [ ]:
models = {}
models['XGBoost'] = xgb_model
models['RandomForest'] = rf_model
models['AdaBoost'] = ada_model
models['SVM'] = svm_model
models['LogisticRegression'] = log_model
models['NaiveBayes'] = nb_model
models['STACKING'] = stacking_model
models['VOTING'] = voting_best

metric = "F1"
candidates = np.linspace(0.05, 0.95, 19)

thr_map, oof_table_rows = {}, []

for name, mdl in models.items():
    s = _oof_scores(mdl, X_tr, y_tr, cv)
    best_t, best_v = None, -1.0
    for t in candidates:
        v = _score_metric(y_tr, (s >= t).astype(int), metric)
        if v > best_v: best_t, best_v = float(t), float(v)
    thr_map[name] = best_t
    oof_table_rows.append({"Model": name.upper(), "Best threshold": best_t, f"OOF {metric}": best_v})

oof_df = pd.DataFrame(oof_table_rows).sort_values(f"OOF {metric}", ascending=False).reset_index(drop=True)
display(oof_df)


NameError: name 'xgb_model' is not defined

## Summary

In [ ]:
# evaluate on test set
test_rows = []
for name, mdl in models.items():
    df_m, _ = evaluate_classification(mdl, X_te, y_te, threshold=thr_map[name], show=False)
    get = lambda k: float(df_m.loc[df_m["Metric"] == k, "Value"])
    test_rows.append({
        "Model": name,
        "thr": t,
        "F1": get("F1"),
        "MCC": get("MCC"),
        "Balanced accuracy": get("Balanced accuracy"),
        "ROC-AUC": get("ROC-AUC"),
        "PR-AUC": get("PR-AUC (Average Precision)"),
        "Accuracy": get("Accuracy"),
    })
test_df = pd.DataFrame(test_rows).sort_values(["F1","ROC-AUC","MCC"], ascending=False).reset_index(drop=True)
display(test_df)

C:\Users\pajak\AppData\Local\Temp\ipykernel_7832\1135036317.py:15: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  return float(dfm.loc[dfm["Metric"]==metric, "Value"])
C:\Users\pajak\AppData\Local\Temp\ipykernel_7832\1135036317.py:15: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  return float(dfm.loc[dfm["Metric"]==metric, "Value"])
C:\Users\pajak\AppData\Local\Temp\ipykernel_7832\1135036317.py:15: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  return float(dfm.loc[dfm["Metric"]==metric, "Value"])
C:\Users\pajak\AppData\Local\Temp\ipykernel_7832\1135036317.py:15: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) 


=== Test set comparison (threshold 0.5 for base models, tuned for STACKING) ===


,Model,Accuracy,Balanced accuracy,F1,ROC-AUC,PR-AUC,MCC
6,STACKING,0.9355,0.9396,0.9091,0.9791,0.9580,0.8614
2,XGBoost,0.9194,0.9042,0.8780,0.9652,0.9133,0.8184
5,LogisticRegression,0.9194,0.9042,0.8780,0.9652,0.9281,0.8184
4,SVM,0.9032,0.8920,0.8571,0.9489,0.9201,0.7840
1,AdaBoost,0.9032,0.8804,0.8500,0.9686,0.9414,0.7810
3,NaiveBayes,0.8871,0.8682,0.8293,0.9303,0.8441,0.7455
7,VOTING,0.8871,0.8566,0.8205,0.9803,0.9606,0.7435
0,RandomForest,0.8871,0.8566,0.8205,0.9779,0.9555,0.7435
